# 総合ランキング公平化 分析

この notebook は、総合ランキングを変更する前に現行式の傾向を確認するための分析用です。

- 現行の総合ランキング式は変更しません。
- 総合ランキングを維持するか、別枠の発掘候補を追加するかは未決定です。
- 候補抽出条件と `discovery_score` は仕様案ではなく、比較用の分析仮説です。
- `data/videos.db` は read-only URI (`mode=ro`) で開き、`PRAGMA query_only=ON` を設定します。
- DB が存在しない場合、ダミーデータは作らず、想定パスを表示して分析セルをスキップします。

In [ ]:
from __future__ import annotations

import sqlite3
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "videos.db"
TOP_N = 50
AVAILABLE_ONLY = True

REQUIRED_TABLES = ("videos", "viewing_history", "judgment_history", "likes")
analysis_ready = DB_PATH.exists()

def open_readonly_db(db_path: Path) -> sqlite3.Connection:
    uri = f"file:{db_path.as_posix()}?mode=ro"
    conn = sqlite3.connect(uri, uri=True)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA query_only=ON")
    return conn

if not analysis_ready:
    print("DB が見つからないため分析をスキップします。")
    print(f"想定パス: {DB_PATH}")
    print("ローカル環境の data/videos.db を配置してから再実行してください。")
else:
    with open_readonly_db(DB_PATH) as conn:
        existing_tables = set(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"])
        missing = sorted(set(REQUIRED_TABLES) - existing_tables)
        if missing:
            analysis_ready = False
            print(f"必要なテーブルが不足しているため分析をスキップします: {missing}")
        else:
            videos = pd.read_sql_query("SELECT * FROM videos", conn)
            viewing_history = pd.read_sql_query("SELECT * FROM viewing_history", conn)
            judgment_history = pd.read_sql_query("SELECT * FROM judgment_history", conn)
            likes = pd.read_sql_query("SELECT * FROM likes", conn)
            print(f"loaded: videos={len(videos):,}, viewing_history={len(viewing_history):,}, judgment_history={len(judgment_history):,}, likes={len(likes):,}")

if not analysis_ready:
    videos = pd.DataFrame()
    viewing_history = pd.DataFrame()
    judgment_history = pd.DataFrame()
    likes = pd.DataFrame()

## 派生列の作成

現行の総合スコア式を notebook 内で再計算します。

`round((view_days*1 + like_count*3) * (1 + 0.5*T1 + 0.3*T2) * 100)`

In [ ]:
def _as_int(series: pd.Series, default: int = 0) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(default).astype(int)

def add_analysis_columns(videos_df: pd.DataFrame) -> pd.DataFrame:
    df = videos_df.copy()
    if df.empty:
        return df

    for col in ("current_favorite_level", "is_available", "is_deleted", "needs_selection", "is_selection_completed"):
        if col not in df.columns:
            df[col] = 0
        df[col] = _as_int(df[col])

    vh = viewing_history.copy()
    if not vh.empty:
        vh["viewed_at"] = pd.to_datetime(vh["viewed_at"], errors="coerce")
        vh["view_date"] = vh["viewed_at"].dt.date
        view_stats = vh.groupby("video_id").agg(
            view_count=("id", "size"),
            view_days=("view_date", "nunique"),
            last_viewed_at=("viewed_at", "max"),
        ).reset_index().rename(columns={"video_id": "id"})
    else:
        view_stats = pd.DataFrame(columns=["id", "view_count", "view_days", "last_viewed_at"])

    lk = likes.copy()
    if not lk.empty:
        lk["liked_at"] = pd.to_datetime(lk["liked_at"], errors="coerce")
        like_stats = lk.groupby("video_id").agg(like_count=("id", "size")).reset_index().rename(columns={"video_id": "id"})
    else:
        like_stats = pd.DataFrame(columns=["id", "like_count"])

    jh = judgment_history.copy()
    if not jh.empty:
        jh["judged_at"] = pd.to_datetime(jh["judged_at"], errors="coerce")
        if "was_selection_judgment" not in jh.columns:
            jh["was_selection_judgment"] = 0
        jh["was_selection_judgment"] = _as_int(jh["was_selection_judgment"])
        judgment_stats = jh.groupby("video_id").agg(
            judgment_count=("id", "size"),
            tier1_judgment_count=("was_selection_judgment", lambda s: int((s == 0).sum())),
            tier2_judgment_count=("was_selection_judgment", lambda s: int((s == 1).sum())),
        ).reset_index().rename(columns={"video_id": "id"})
    else:
        judgment_stats = pd.DataFrame(columns=["id", "judgment_count", "tier1_judgment_count", "tier2_judgment_count"])

    for stats in (view_stats, like_stats, judgment_stats):
        df = df.merge(stats, on="id", how="left")

    count_cols = ["view_count", "view_days", "like_count", "judgment_count", "tier1_judgment_count", "tier2_judgment_count"]
    for col in count_cols:
        df[col] = _as_int(df[col]) if col in df.columns else 0

    df["last_viewed_at"] = pd.to_datetime(df.get("last_viewed_at"), errors="coerce")
    today = pd.Timestamp.now().normalize()
    df["days_since_last_view"] = (today - df["last_viewed_at"].dt.normalize()).dt.days
    df["file_size_gb"] = pd.to_numeric(df.get("file_size"), errors="coerce") / (1024 ** 3)
    df["is_judged"] = df["current_favorite_level"] >= 0
    df["tier1_unrated"] = (df["current_favorite_level"] == -1) & (df["needs_selection"] == 0) & (df["is_selection_completed"] == 0)
    df["tier2_unselected"] = (df["needs_selection"] == 1) & (df["is_selection_completed"] == 0)

    t1 = df["is_judged"].astype(int)
    t2 = df["is_selection_completed"].astype(int)
    base = df["view_days"] * 1 + df["like_count"] * 3
    bonus = 1 + 0.5 * t1 + 0.3 * t2
    df["composite_score"] = (base * bonus * 100).round(0).astype(int)
    return df

analysis_df = add_analysis_columns(videos)
analysis_df.head()

## データ品質チェック

In [ ]:
if analysis_df.empty:
    print("分析対象データがありません。")
else:
    quality_summary = pd.DataFrame([
        {"metric": "videos", "value": len(analysis_df)},
        {"metric": "is_deleted=0", "value": int((analysis_df["is_deleted"] == 0).sum())},
        {"metric": "is_available=1", "value": int((analysis_df["is_available"] == 1).sum())},
        {"metric": "essential_filename duplicated", "value": int(analysis_df["essential_filename"].duplicated().sum()) if "essential_filename" in analysis_df.columns else None},
        {"metric": "current_full_path missing", "value": int(analysis_df.get("current_full_path", pd.Series(dtype=object)).isna().sum())},
        {"metric": "未判定", "value": int((analysis_df["current_favorite_level"] == -1).sum())},
        {"metric": "Tier2 未選別", "value": int(analysis_df["tier2_unselected"].sum())},
        {"metric": "いいね済み", "value": int((analysis_df["like_count"] > 0).sum())},
        {"metric": "再生済み", "value": int((analysis_df["view_count"] > 0).sum())},
        {"metric": "最終再生日なし", "value": int(analysis_df["last_viewed_at"].isna().sum())},
    ])
    display(quality_summary)
    display(analysis_df["current_favorite_level"].value_counts(dropna=False).sort_index().rename_axis("level").reset_index(name="count"))
    display(analysis_df.get("storage_location", pd.Series(dtype=object)).fillna("(missing)").value_counts().rename_axis("storage_location").reset_index(name="count"))

## 総合ランキング上位分析

既定は全期間、利用可能のみ、Top 50 です。比較表では 180日、1年、全期間を並べます。

In [ ]:
def ranked_for_period(days: int | None, top_n: int = TOP_N, available_only: bool = AVAILABLE_ONLY) -> pd.DataFrame:
    if analysis_df.empty:
        return pd.DataFrame()
    df = analysis_df[analysis_df["is_deleted"] == 0].copy()
    if available_only:
        df = df[df["is_available"] == 1].copy()
    if days is not None:
        start = pd.Timestamp.now() - pd.Timedelta(days=days)
        vh = viewing_history.copy()
        lk = likes.copy()
        vh["viewed_at"] = pd.to_datetime(vh.get("viewed_at"), errors="coerce")
        lk["liked_at"] = pd.to_datetime(lk.get("liked_at"), errors="coerce")
        vh = vh[vh["viewed_at"] >= start]
        lk = lk[lk["liked_at"] >= start]
        view_days = vh.assign(view_date=vh["viewed_at"].dt.date).groupby("video_id")["view_date"].nunique().rename("view_days_period")
        like_count = lk.groupby("video_id").size().rename("like_count_period")
        df = df.merge(view_days, left_on="id", right_index=True, how="left")
        df = df.merge(like_count, left_on="id", right_index=True, how="left")
        df["view_days_period"] = _as_int(df["view_days_period"])
        df["like_count_period"] = _as_int(df["like_count_period"])
    else:
        df["view_days_period"] = df["view_days"]
        df["like_count_period"] = df["like_count"]
    t1 = (df["current_favorite_level"] >= 0).astype(int)
    t2 = df["is_selection_completed"].astype(int)
    df["score_period"] = ((df["view_days_period"] + df["like_count_period"] * 3) * (1 + 0.5 * t1 + 0.3 * t2) * 100).round(0).astype(int)
    df = df[df["score_period"] > 0].sort_values(["score_period", "last_viewed_at"], ascending=[False, False], na_position="last")
    df.insert(0, "rank", range(1, len(df) + 1))
    return df.head(top_n)

top_all = ranked_for_period(None)
ranking_cols = ["rank", "essential_filename", "current_favorite_level", "score_period", "view_days_period", "like_count_period", "tier1_judgment_count", "tier2_judgment_count", "last_viewed_at", "storage_location", "file_size_gb"]
display(top_all[[c for c in ranking_cols if c in top_all.columns]])

period_compare = []
for label, days in (("180日", 180), ("1年", 365), ("全期間", None)):
    ranked = ranked_for_period(days)
    period_compare.append({
        "period": label,
        "items": len(ranked),
        "median_score": ranked["score_period"].median() if not ranked.empty else None,
        "median_view_days": ranked["view_days_period"].median() if not ranked.empty else None,
        "median_like_count": ranked["like_count_period"].median() if not ranked.empty else None,
        "tier2_completed_rate": ranked["is_selection_completed"].mean() if not ranked.empty else None,
    })
display(pd.DataFrame(period_compare))

## 二極化確認

`総合Top50`、`非Top50`、`score=0` で再生済み率、いいね率、判定済み率、Tier2選別済み率、最終再生日、視聴日数を比較します。

In [ ]:
if analysis_df.empty:
    print("分析対象データがありません。")
else:
    scope = analysis_df[(analysis_df["is_deleted"] == 0) & ((analysis_df["is_available"] == 1) if AVAILABLE_ONLY else True)].copy()
    top_ids = set(top_all["id"].tolist()) if not top_all.empty else set()
    scope["segment"] = "非Top50"
    scope.loc[scope["id"].isin(top_ids), "segment"] = "総合Top50"
    scope.loc[scope["composite_score"] == 0, "segment"] = "score=0"
    segment_summary = scope.groupby("segment").agg(
        count=("id", "size"),
        played_rate=("view_count", lambda s: float((s > 0).mean())),
        liked_rate=("like_count", lambda s: float((s > 0).mean())),
        judged_rate=("is_judged", "mean"),
        tier2_completed_rate=("is_selection_completed", "mean"),
        median_days_since_last_view=("days_since_last_view", "median"),
        median_view_days=("view_days", "median"),
        median_score=("composite_score", "median"),
    ).reset_index()
    display(segment_summary)

## 発掘候補の仮説抽出

A-E は仕様ではなく、総合ランキング外に置く別枠候補の比較用仮説です。既定表示列には `current_full_path` を含めません。

In [ ]:
candidate_cols = [
    "essential_filename", "current_favorite_level", "composite_score", "view_count", "view_days", "like_count",
    "judgment_count", "tier1_judgment_count", "tier2_judgment_count", "needs_selection", "is_selection_completed",
    "last_viewed_at", "days_since_last_view", "storage_location", "file_size_gb",
]

def show_candidates(name: str, df: pd.DataFrame, limit: int = 30) -> None:
    print(f"{name}: {len(df):,}件")
    if not df.empty:
        display(df[[c for c in candidate_cols if c in df.columns]].head(limit))

if analysis_df.empty:
    print("分析対象データがありません。")
else:
    base = analysis_df[(analysis_df["is_deleted"] == 0) & ((analysis_df["is_available"] == 1) if AVAILABLE_ONLY else True)].copy()
    top_ids = set(top_all["id"].tolist()) if not top_all.empty else set()
    non_top = base[~base["id"].isin(top_ids)].copy()
    q75 = non_top.loc[non_top["composite_score"] > 0, "composite_score"].quantile(0.75)
    med_view = non_top["view_count"].median()
    med_judgment = non_top["judgment_count"].median()

    cand_a = non_top[(non_top["composite_score"] > 0) & (non_top["composite_score"] >= q75) & ((non_top["view_count"] <= med_view) | (non_top["judgment_count"] <= med_judgment))].sort_values("composite_score", ascending=False)
    cand_b = base[(base["current_favorite_level"] >= 0) & (base["needs_selection"] == 1) & (base["is_selection_completed"] == 0)].sort_values(["current_favorite_level", "composite_score"], ascending=False)
    cand_c = non_top[(non_top["like_count"] > 0) | (non_top["current_favorite_level"] >= 3) | (non_top["view_days"] >= 2) | (non_top["is_selection_completed"] == 1)].sort_values("composite_score", ascending=False)
    good_signal = (base["current_favorite_level"] >= 3) | (base["like_count"] > 0) | (base["view_days"] >= 2) | (base["is_selection_completed"] == 1)
    stale_signal = base["last_viewed_at"].isna() | (base["days_since_last_view"] >= 90)
    cand_d = base[stale_signal & good_signal].sort_values(["days_since_last_view", "composite_score"], ascending=[False, False], na_position="first")

    score_max = max(float(base["composite_score"].max()), 1.0)
    basic = (base["composite_score"] / score_max).clip(0, 1)
    underexposed = (1 - (base["view_count"] / max(float(base["view_count"].quantile(0.90)), 1.0))).clip(0, 1)
    unprocessed = ((base["tier1_unrated"].astype(int) + base["tier2_unselected"].astype(int)) / 2).clip(0, 1)
    stale = ((base["days_since_last_view"].fillna(365) / 365).clip(0, 1))
    cand_e = base.copy()
    cand_e["discovery_score"] = 0.45 * basic + 0.20 * underexposed + 0.20 * unprocessed + 0.15 * stale
    cand_e = cand_e[~cand_e["id"].isin(top_ids)].sort_values("discovery_score", ascending=False)

    show_candidates("A: 非Top50・高スコアだが露出/処理が少ない", cand_a)
    show_candidates("B: Lv判定済みだがTier2未選別", cand_b)
    show_candidates("C: 非Top50の良い兆候あり", cand_c)
    show_candidates("D: 未再生/90日以上前だが良い兆候あり", cand_d)
    show_candidates("E: discovery_score 仮説", cand_e[[*base.columns, "discovery_score"]])

## まとめメモ

実行後、以下を確認する。

- 総合Top50が視聴日数・いいね・T1/T2ボーナスのどれに強く寄っているか。
- `score=0` と `非Top50` に、感覚的には拾いたい作品が残っていないか。
- A-E のうち、候補数と内容が実運用の確認量として妥当か。
- `discovery_score` は仕様ではなく仮説なので、採用する場合は係数・上限・表示場所を別途仕様化する。
- 個人パスや巨大な候補一覧はドキュメント化しない。